# Experiment 11: Customer Review Analysis Using NLP

**Aim:** Build an NLP system to preprocess customer reviews, extract linguistic features, analyze sentiment, find collocations/key phrases, measure text similarity, and compare ratings with sentiment.

**Lab manual:** Vardhaman College NLP Lab (A8708, R22).

Run the setup/download cell if required. Expected outputs may vary slightly with NLTK/spaCy dataset or model versions.

This is the case-study program included in the lab manual, structured as a runnable notebook. Install the listed dependencies if they are not available.

In [ ]:
# If needed, install dependencies in a notebook cell:
# %pip install nltk pandas numpy matplotlib

import re
import string
import nltk
import numpy as np
import pandas as pd
from collections import Counter, defaultdict
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk import pos_tag
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.sentiment import SentimentIntensityAnalyzer
from nltk.collocations import BigramCollocationFinder, BigramAssocMeasures
from nltk.util import bigrams, trigrams

for package in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4",
                "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng",
                "vader_lexicon"]:
    nltk.download(package, quiet=True)

def generate_ngrams(text):
    tokens = [w for w in word_tokenize(text.lower()) if w.isalpha()]
    return Counter(bigrams(tokens)), Counter(trigrams(tokens))

class CustomerReviewAnalyzer:
    def __init__(self):
        self.lemmatizer = WordNetLemmatizer()
        self.stop_words = set(stopwords.words("english"))
        self.sentiment_analyzer = SentimentIntensityAnalyzer()

    def load_data(self):
        reviews = [
            {"review": "This product is absolutely amazing! Great quality and fast delivery.", "rating": 5, "product_category": "Electronics"},
            {"review": "Poor quality item. Broke after just one week of use. Very disappointed.", "rating": 1, "product_category": "Electronics"},
            {"review": "Good value for money. The product works well and shipping was quick.", "rating": 4, "product_category": "Home & Garden"},
            {"review": "Average product. Nothing special but does the job. Could be better.", "rating": 3, "product_category": "Electronics"},
            {"review": "Excellent customer service! The team was very helpful and responsive.", "rating": 5, "product_category": "Books"},
            {"review": "Terrible experience. The item was damaged and customer service was unhelpful.", "rating": 1, "product_category": "Books"},
            {"review": "Love this product! It exceeded my expectations. Highly recommended!", "rating": 5, "product_category": "Home & Garden"},
            {"review": "Decent product but overpriced. There are better alternatives available.", "rating": 2, "product_category": "Electronics"},
        ]
        self.df = pd.DataFrame(reviews)
        return self.df

    def preprocess_text(self, text):
        text = text.lower()
        text = re.sub(r"\d+", "", text)
        text = text.translate(str.maketrans("", "", string.punctuation))
        tokens = word_tokenize(text)
        tokens = [w for w in tokens if w.isalpha() and w not in self.stop_words]
        return [self.lemmatizer.lemmatize(w) for w in tokens]

    def extract_features(self, text):
        tokens = word_tokenize(text)
        tags = pos_tag(tokens)
        return {
            "word_count": len(tokens),
            "sentence_count": len(sent_tokenize(text)),
            "pos_counts": dict(Counter(tag for _, tag in tags)),
            "sentiment": self.sentiment_analyzer.polarity_scores(text)
        }

    def run(self):
        df = self.load_data()
        print("CUSTOMER REVIEW ANALYSIS - NLP CASE STUDY")
        print("=" * 55)
        print("Total reviews:", len(df))
        print("Product categories:", list(df["product_category"].unique()))
        print("\nRating distribution:")
        print(df["rating"].value_counts().sort_index())

        features = []
        processed = []
        for i, review in enumerate(df["review"], start=1):
            tokens = self.preprocess_text(review)
            feat = self.extract_features(review)
            processed.append(tokens)
            features.append(feat)
            print(f"\nReview {i}: {review}")
            print("Processed:", " ".join(tokens))
            print("Word count:", feat["word_count"], "| Sentences:", feat["sentence_count"])
            print("Sentiment compound:", round(feat["sentiment"]["compound"], 2))

        all_words = [w for tokens in processed for w in tokens]
        word_freq = Counter(all_words)
        print("\nTop words:", word_freq.most_common(10))

        category_scores = defaultdict(list)
        for i, row in df.iterrows():
            category_scores[row["product_category"]].append(features[i]["sentiment"]["compound"])
        print("\nAverage sentiment by category:")
        for category, scores in category_scores.items():
            print(f"{category}: {np.mean(scores):.3f}")

        # Adjective + noun phrase extraction
        phrases = []
        for review in df["review"]:
            tagged = pos_tag(word_tokenize(review))
            for i in range(len(tagged)-1):
                if tagged[i][1].startswith("JJ") and tagged[i+1][1].startswith("NN"):
                    phrases.append(tagged[i][0] + " " + tagged[i+1][0])
        print("\nCommon adjective-noun phrases:", Counter(phrases).most_common(5))

        # Jaccard similarity for first four reviews
        print("\nJaccard similarity matrix (first 4 reviews):")
        token_sets = [set(self.preprocess_text(x)) for x in df["review"][:4]]
        for a in token_sets:
            row = []
            for b in token_sets:
                union = a | b
                row.append(len(a & b) / len(union) if union else 0)
            print([round(x, 3) for x in row])

        ratings = df["rating"].to_numpy()
        sentiments = np.array([f["sentiment"]["compound"] for f in features])
        corr = np.corrcoef(ratings, sentiments)[0, 1]
        print("\nRating-sentiment correlation:", round(float(corr), 3))

        all_text = " ".join(df["review"])
        bigram_freq, trigram_freq = generate_ngrams(all_text)
        print("\nTop bigrams:", bigram_freq.most_common(3))
        print("Top trigrams:", trigram_freq.most_common(3))
        print("\nSummary:")
        print(f"Average rating: {df['rating'].mean():.2f}/5")
        print(f"Average compound sentiment: {sentiments.mean():.3f}")
        print("Most common word:", word_freq.most_common(1)[0][0])
        print("Most positive category:", max(category_scores, key=lambda k: np.mean(category_scores[k])))

        return df, features, word_freq

analyzer = CustomerReviewAnalyzer()
df, features, word_frequency = analyzer.run()


## Expected output

```text
CUSTOMER REVIEW ANALYSIS - NLP CASE STUDY
=======================================================
Total reviews: 8
Product categories: ['Electronics', 'Home & Garden', 'Books']

Rating distribution:
rating
1    2
2    1
3    1
4    1
5    3

The notebook then prints each review, its cleaned tokens, word/sentence counts,
VADER sentiment scores, frequent words, average sentiment by category,
adjective-noun phrases, a Jaccard similarity matrix, rating-sentiment correlation,
top bigrams/trigrams, and summary statistics.

Exact sentiment scores and some ranked outputs depend on the installed NLTK/VADER version.
```
